# DRAFT — 04c human immune de novo annotation

**Presentation scaffold only.** Generated 2026-08-12 from the committed decision
record `/n/lab_storage/eddy_lab/users/nhilgert/xmens_final/resources/annotations/human_immune.csv` and locked vocabulary `/n/lab_storage/eddy_lab/users/nhilgert/xmens_final/resources/annotation_vocabulary.yaml`. The map below is
frozen from the CSV; this notebook does not infer, edit, or write annotations.

- resolution: `leiden_scVI_3.0`
- decision CSV SHA-256: `8c24e61758fbc590e1bd8393fad26d028dbd6239edd024b065c57162474f2433`
- vocabulary SHA-256: `d02b68120a62d1764b6f0473976fcd72a85f5c30a081b3a857db9215e6a38847`


## 1. Frozen configuration and annotation map

In [ ]:
SPECIES = 'human'
LINEAGE = 'immune'
RES = 'leiden_scVI_3.0'
REPO = '/n/lab_storage/eddy_lab/users/nhilgert/xmens_final'
CSV_PATH = '/n/lab_storage/eddy_lab/users/nhilgert/xmens_final/resources/annotations/human_immune.csv'
VOCAB_PATH = '/n/lab_storage/eddy_lab/users/nhilgert/xmens_final/resources/annotation_vocabulary.yaml'
SPEC_PATH = '/n/lab_storage/eddy_lab/users/nhilgert/xmens_final/resources/lineage_spec.yaml'
LATENT_PATH = '/n/lab_storage/eddy_lab/users/nhilgert/xmens_final/data/interim/latent_human_immune.h5ad'
V2_PATH = '/n/lab_storage/eddy_lab/users/nhilgert/xmens_final/data/interim/migrated/xmens_annotation_v2_20260811/reannotated_migrated/human_immune.h5ad'
PRODUCER_PATH = '/n/lab_storage/eddy_lab/users/nhilgert/xmens_final/workflow/scripts/apply_annotation.py'
STATUS_PATH = '/n/lab_storage/eddy_lab/users/nhilgert/xmens_final/data/interim/migrated/xmens_annotation_v2_20260811/manifests/MIGRATION_STATUS.json'
READY_PATH = '/n/lab_storage/eddy_lab/users/nhilgert/xmens_final/data/interim/migrated/xmens_annotation_v2_20260811/MIGRATION_REVIEW_READY'
CSV_SHA256 = '8c24e61758fbc590e1bd8393fad26d028dbd6239edd024b065c57162474f2433'
VOCAB_SHA256 = 'd02b68120a62d1764b6f0473976fcd72a85f5c30a081b3a857db9215e6a38847'
SPEC_SHA256 = '63f2df33bf79b9db09413ba5f4c507318ea30f2001f739cb90292a05f516426f'
LATENT_SHA256 = '50c0801938ed18e61a26f9d3f6a241106f7e9a5f2fc194984ac7785d0397b2a5'
V2_SHA256 = '810f2a391cac2d3065c06f7690841bc7b7ce9a437de75838e294c0c279f59575'
PRODUCER_SHA256 = 'e57cd6e6b66ece138e770048060e7eece2eb6020395bc21b73aab4e78812b480'
STATUS_SHA256 = 'def88e5240f653f98c8d78adf7f8dddcdb05bfda30d08bd0fada7df489969208'
READY_SHA256 = 'c26de83abdc9496cd1301470918ec39ecca1cf389ef0ae1c6504da1800d1c431'
GENERATOR_SHA256 = '82c7709ecfead2dd17ffe10ea4eb346c6e7cbcec8f048d4a2b259f2cabfb5a11'
ANNOTATION = {
    "0": "macrophage",
    "1": "EXCLUDE",
    "2": "macrophage",
    "3": "EXCLUDE",
    "4": "macrophage",
    "5": "t_cell_cd8",
    "6": "unk_cell_gzmb_high_itgax_high",
    "7": "t_cell_gamma_delta",
    "8": "unk_cell_gzmb_low_itgax_low",
    "9": "EXCLUDE",
    "10": "t_cell_cd8",
    "11": "macrophage",
    "12": "t_cell_cd8",
    "13": "mast_cell",
    "14": "t_cell_regulatory",
    "15": "t_cell_cd4",
    "16": "t_cell_regulatory",
    "17": "t_cell_cd4",
    "18": "t_cell_cd4",
    "19": "t_cell_cd4",
    "20": "t_cell_cd4",
    "21": "unk_cell_gzmb_high_itgax_high",
    "22": "t_cell_cd8",
    "23": "t_cell_cd8",
    "24": "t_cell_cd8",
    "25": "t_cell_cd8",
    "26": "t_cell_cd8",
    "27": "t_cell_cd8"
}
MARKERS = {
    "pan": [
        "PTPRC"
    ],
    "myeloid": [
        "CD68",
        "CSF1R",
        "ITGAM",
        "LYZ",
        "ITGAX"
    ],
    "lymph": [
        "CD3E",
        "CD8A",
        "CD4",
        "FOXP3"
    ],
    "nk": [
        "NCAM1",
        "NCR1",
        "KLRD1",
        "GZMB"
    ],
    "mast": [
        "TPSAB1",
        "CPA3",
        "MS4A2",
        "KIT"
    ],
    "contam": [
        "CDH1",
        "PDGFRA",
        "PECAM1",
        "ACTA2"
    ]
}

## 2. Load the frozen lineage latent and verify complete cluster coverage

In [ ]:
import hashlib, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import pandas as pd, scanpy as sc

REPO = Path(REPO)
def _sha(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()
assert _sha(LATENT_PATH) == LATENT_SHA256, (
    "latent changed; regenerate this draft")
adata = sc.read_h5ad(LATENT_PATH)
assert RES in adata.obs, f"{RES} absent from {LATENT_PATH}"
clusters = set(adata.obs[RES].astype(str).unique())
assert clusters == set(ANNOTATION), (
    f"frozen map mismatch: missing={sorted(clusters-set(ANNOTATION))}, "
    f"extra={sorted(set(ANNOTATION)-clusters)}")
def _cluster_order(value):
    try:
        return (0, float(value), str(value))
    except ValueError:
        return (1, 0.0, str(value))
cluster_order = sorted(clusters, key=_cluster_order)
adata.obs[RES] = pd.Categorical(
    adata.obs[RES].astype(str), categories=cluster_order, ordered=True)
assert list(adata.obs[RES].cat.categories) == cluster_order
print("numeric cluster order:", cluster_order)
print(f"{adata.n_obs:,} cells x {adata.n_vars:,} genes; {len(clusters)} clusters")

## 3. Mechanical marker-group scores and RAW marker dotplot by frozen cluster

In [ ]:
import numpy as np
plot_sets = {name: [gene for gene in genes if gene in adata.var_names]
             for name, genes in MARKERS.items()}
plot_sets = {name: genes for name, genes in plot_sets.items() if genes}
missing_groups = sorted(set(MARKERS) - set(plot_sets))
assert not missing_groups, f"configured marker groups absent from latent: {missing_groups}"
for name, genes in MARKERS.items():
    missing_genes = [gene for gene in genes if gene not in adata.var_names]
    if missing_genes:
        print(f"marker genes absent for {name}: {missing_genes}")
# Existing 04c convention: each marker-group score is the per-cell mean
# of its present marker genes. Scores are printed for review and never
# used to alter the CSV-derived ANNOTATION map.
group_scores = pd.DataFrame(index=adata.obs_names)
for name, genes in plot_sets.items():
    group_scores[f"score_{name}"] = np.asarray(
        adata[:, genes].X.mean(axis=1)).ravel()
cluster_scores = group_scores.join(
    adata.obs[[RES]].astype(str)).groupby(RES, sort=False).mean()
cluster_scores = cluster_scores.reindex(cluster_order)
print("marker-group mean scores by numeric cluster order:")
print(cluster_scores.round(3).to_string())
sc.pl.dotplot(adata, plot_sets, groupby=RES, standard_scale="var",
              figsize=(10, max(6, 0.22 * len(clusters))), show=True)

## 4. Apply the committed map in memory using the shared producer

In [ ]:
import sys
assert _sha(PRODUCER_PATH) == PRODUCER_SHA256, (
    "annotation producer changed; regenerate this draft")
sys.path.insert(0, str(REPO / "workflow/scripts"))
from apply_annotation import (apply_map, build_vocabulary_rollup,
                              validate_canonical_annotations)
import yaml

vocab = yaml.safe_load(Path(VOCAB_PATH).read_text())
rollup = build_vocabulary_rollup(vocab)
summary = apply_map(adata, RES, ANNOTATION, rollup=rollup, output_mode="fine")
kept = adata[~adata.obs.excluded].copy()
validate_canonical_annotations(kept, vocab, context=f"{SPECIES}/{LINEAGE} draft")
kept.obs["display_label"] = kept.obs.cell_subtype.astype(str)
v2 = sc.read_h5ad(V2_PATH, backed="r")
try:
    assert kept.shape == v2.shape, "in-memory retained shape differs from certified v2"
    assert kept.obs_names.equals(v2.obs_names), "in-memory retained IDs/order differ from certified v2"
    assert kept.var_names.equals(v2.var_names), "in-memory genes/order differ from certified v2"
    assert kept.obs.cell_type.astype(str).tolist() == v2.obs.cell_type.astype(str).tolist(), (
        "in-memory cell_type differs from certified v2")
    assert kept.obs.cell_subtype.astype(str).tolist() == v2.obs.cell_subtype.astype(str).tolist(), (
        "in-memory cell_subtype differs from certified v2")
    assert not kept.obs.cell_subtype.astype(str).str.strip().eq("").any(), "empty canonical subtype"
finally:
    if getattr(v2, "file", None) is not None:
        v2.file.close()
print(summary.to_string())
print("\ncanonical cell types:")
print(kept.obs.cell_type.value_counts().to_string())
print("\ncanonical subtypes (nonempty):")
print(kept.obs.loc[kept.obs.cell_subtype.ne(""), "cell_subtype"].value_counts().to_string())

## 5. Verification dotplot ordered by the frozen annotation

In [ ]:
if plot_sets:
    sc.pl.dotplot(kept, plot_sets, groupby="display_label", standard_scale="var",
                  figsize=(10, max(6, 0.35 * kept.obs.display_label.nunique())), show=True)
sc.pl.embedding(kept, "X_umap_scVI", color=[RES, "display_label"],
                legend_loc="right margin", show=True)

## 6. Freeze check — assert the committed inputs are unchanged

In [ ]:
assert _sha(CSV_PATH) == CSV_SHA256, "decision CSV changed; regenerate this draft"
assert _sha(VOCAB_PATH) == VOCAB_SHA256, "vocabulary changed; regenerate this draft"
assert _sha(SPEC_PATH) == SPEC_SHA256, "lineage spec changed; regenerate this draft"
assert _sha(LATENT_PATH) == LATENT_SHA256, "latent changed; regenerate this draft"
assert _sha(PRODUCER_PATH) == PRODUCER_SHA256, "annotation producer changed; regenerate this draft"
assert _sha(V2_PATH) == V2_SHA256, "certified v2 lineage changed; regenerate this draft"
assert _sha(STATUS_PATH) == STATUS_SHA256, "v2 status changed; regenerate this draft"
assert _sha(READY_PATH) == READY_SHA256, "v2 readiness changed; regenerate this draft"
now = pd.read_csv(CSV_PATH, dtype=str)
assert not now[["cluster", "label"]].isna().any().any(), "blank CSV value"
assert not now.cluster.str.strip().duplicated().any(), "duplicate CSV cluster"
now_map = dict(zip(now.cluster.str.strip(), now.label.str.strip()))
assert now_map == ANNOTATION, "embedded map differs from committed CSV"
print(now.to_string(index=False))
print("PASS: CSV and vocabulary hashes match; embedded map is exact.")